# Классификация CC50 > median

Бинарная классификация: CC50 выше медианы выборки


In [1]:
import numpy as np
import pickle
RANDOM_STATE = 42
ASSETS_DIR = 'assets'
try:
    import seaborn as sns
    sns.set_style('whitegrid')
except ImportError:
    pass


In [2]:
import os
os.makedirs(ASSETS_DIR, exist_ok=True)
with open('_data.pkl', 'rb') as f:
    data = pickle.load(f)
df = data['df']
feat_cols = data['feat_cols']
target_cols = data['target_cols']
with open('_all_results.pkl', 'rb') as f:
    results = pickle.load(f)
clf_results = results['clf_results']
print(f'Данные: {df.shape}, Признаки: {len(feat_cols)}')


Данные: (1001, 195), Признаки: 192


In [3]:
task_name = '06 Classification Cc50 Median'
target_col = 'CC50, mM'
threshold = np.median(df[target_col].values)
y_bin = (df[target_col].values > threshold).astype(int)
n_pos = int(y_bin.sum())
n_neg = len(y_bin) - n_pos
print(f'CC50, mM > {threshold:.2f} (медиана)')
print(f'Positive: {n_pos}, Negative: {n_neg}, Ratio: {n_pos/len(y_bin):.3f}')


CC50, mM > 411.04 (медиана)
Positive: 499, Negative: 502, Ratio: 0.499


In [4]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, average_precision_score)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all = df[feat_cols].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X_all)


In [5]:
models = [
    ('DummyStratified', DummyClassifier, {'strategy': 'stratified', 'random_state': RANDOM_STATE}),
    ('DummyMostFrequent', DummyClassifier, {'strategy': 'most_frequent'}),
    ('LogReg', LogisticRegression, {'max_iter': 1000, 'random_state': RANDOM_STATE, 'class_weight': 'balanced'}),
    ('KNN', KNeighborsClassifier, {'n_neighbors': 5}),
    ('RF', RandomForestClassifier, {'n_estimators': 100, 'random_state': RANDOM_STATE, 'class_weight': 'balanced'}),
    ('HGB', GradientBoostingClassifier, {'n_estimators': 200, 'max_depth': 5, 'random_state': RANDOM_STATE}),
]

print(f'Модели для {task_name}:')
for name, ModelClass, kwargs in models:
    m = ModelClass(**kwargs)
    m.fit(Xs, y_bin)
    preds = m.predict(Xs)
    probs = m.predict_proba(Xs)[:, 1] if hasattr(m, 'predict_proba') else [0]*len(y_bin)
    acc_cv = cross_val_score(m, Xs, y_bin, cv=kf, scoring='accuracy')
    f1_cv = cross_val_score(m, Xs, y_bin, cv=kf, scoring='f1')
    roc_cv = cross_val_score(m, Xs, y_bin, cv=kf, scoring='roc_auc')
    pr_cv = cross_val_score(m, Xs, y_bin, cv=kf, scoring='average_precision')
    print(f'  {name}: acc={np.mean(acc_cv):.3f}(+/-{np.std(acc_cv):.3f}), ROC-AUC={np.mean(roc_cv):.3f}, PR-AUC={np.mean(pr_cv):.3f}')


Модели для 06 Classification Cc50 Median:
  DummyStratified: acc=0.482(+/-0.020), ROC-AUC=0.482, PR-AUC=0.490
  DummyMostFrequent: acc=0.501(+/-0.002), ROC-AUC=0.500, PR-AUC=0.499
  LogReg: acc=0.736(+/-0.034), ROC-AUC=0.816, PR-AUC=0.806
  KNN: acc=0.733(+/-0.031), ROC-AUC=0.825, PR-AUC=0.802
  RF: acc=0.766(+/-0.029), ROC-AUC=0.844, PR-AUC=0.847
  HGB: acc=0.751(+/-0.035), ROC-AUC=0.837, PR-AUC=0.841


In [6]:
best = max(clf_results['CC50 > median'], key=lambda k: clf_results['CC50 > median'][k]['roc_auc_cv'])
r = clf_results['CC50 > median'][best]
print(f'\nЛучшая модель для CC50 > median: {best}')
print(f'  ROC-AUC CV: {r["roc_auc_cv"]:.4f} (+/-{r["roc_auc_cv_std"]:.4f})')
print(f'  PR-AUC CV: {r["pr_auc_cv"]:.4f} (+/-{r["pr_auc_cv_std"]:.4f})')
print(f'  F1 CV: {r["f1_cv"]:.4f} (+/-{r["f1_cv_std"]:.4f})')



Лучшая модель для CC50 > median: RF
  ROC-AUC CV: 0.8438 (+/-0.0221)
  PR-AUC CV: 0.8473 (+/-0.0201)
  F1 CV: 0.7680 (+/-0.0285)
